# B2-019-attention-transformers — Practice p06 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** matrix-transpose, query-key-value-attention, scaled-dot-product-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The score matrix is Q @ K.T divided by sqrt(d_k). Subtracting each row maximum before exponentiation gives a stable row softmax; the returned output is weights @ V. Boundary checks reject nonnumeric, nonfinite, empty, rank-mismatched, or dimension-incompatible inputs.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10

def scaled_dot_product_attention_np(q, k, v):
    q = np.asarray(q)
    k = np.asarray(k)
    v = np.asarray(v)
    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2:
        raise ValueError("q, k, and v must all be rank two")
    if 0 in q.shape or 0 in k.shape or 0 in v.shape:
        raise ValueError("all axes must be nonempty")
    if q.dtype.kind not in "biufc" or k.dtype.kind not in "biufc" or v.dtype.kind not in "biufc":
        raise TypeError("q, k, and v must be numeric")
    if q.shape[1] != k.shape[1] or k.shape[0] != v.shape[0]:
        raise ValueError("incompatible attention dimensions")
    q = q.astype(np.float64, copy=False)
    k = k.astype(np.float64, copy=False)
    v = v.astype(np.float64, copy=False)
    if not np.all(np.isfinite(q)) or not np.all(np.isfinite(k)) or not np.all(np.isfinite(v)):
        raise ValueError("q, k, and v must be finite")
    scores = (q @ k.T) / np.sqrt(q.shape[1])
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    numerators = np.exp(shifted)
    weights = numerators / np.sum(numerators, axis=-1, keepdims=True)
    output = weights @ v
    return weights, output

q = np.array([[1.0, 0.0]], dtype=np.float64)
k = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=np.float64)
v = np.array([[2.0, 0.0], [0.0, 4.0]], dtype=np.float64)
weights, output = scaled_dot_product_attention_np(q, k, v)
EXPECTED_WEIGHTS = np.array([[0.6697615493266569, 0.3302384506733431]], dtype=np.float64)
EXPECTED_OUTPUT = np.array([[1.3395230986533138, 1.3209538026933725]], dtype=np.float64)

### Answer check

In [ ]:
assert weights.shape == (1, 2) and output.shape == (1, 2)
assert weights.dtype == output.dtype == np.float64
np.testing.assert_allclose(weights, EXPECTED_WEIGHTS, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(output, EXPECTED_OUTPUT, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(weights.sum(axis=-1), np.ones(1), atol=ATOL, rtol=RTOL)